### 분류 문제
- 사용할 데이터셋 : data 폴더 안에 parkinsons.csv
- 사용할 모델 : RandomForest, 다중 퍼셉트론
- 종속 변수 : status 컬럼 (0, 1)
- train, test 데이터셋으로 8:2를 기준으로 데이터를 학습하고 검증
- 다중퍼셉트론의 구조는 정확도 85% 이상 (epoch의 횟수는 150회 제한)

In [8]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier

In [13]:
df = pd.read_csv('../data/parkinsons.csv')

In [29]:
df.head(5)

,MDVP:Fo(Hz),MDVP:Fhi(Hz),MDVP:Flo(Hz),MDVP:Jitter(%),MDVP:Jitter(Abs),MDVP:RAP,MDVP:PPQ,Jitter:DDP,MDVP:Shimmer,MDVP:Shimmer(dB),...,Shimmer:DDA,NHR,HNR,status,RPDE,DFA,spread1,spread2,D2,PPE
0,119.992,157.302,74.997,0.00784,0.00007,0.00370,0.00554,0.01109,0.04374,0.426,...,0.06545,0.02211,21.033,1,0.414783,0.815285,-4.813031,0.266482,2.301442,0.284654
1,122.400,148.650,113.819,0.00968,0.00008,0.00465,0.00696,0.01394,0.06134,0.626,...,0.09403,0.01929,19.085,1,0.458359,0.819521,-4.075192,0.335590,2.486855,0.368674
2,116.682,131.111,111.555,0.01050,0.00009,0.00544,0.00781,0.01633,0.05233,0.482,...,0.08270,0.01309,20.651,1,0.429895,0.825288,-4.443179,0.311173,2.342259,0.332634
3,116.676,137.871,111.366,0.00997,0.00009,0.00502,0.00698,0.01505,0.05492,0.517,...,0.08771,0.01353,20.644,1,0.434969,0.819235,-4.117501,0.334147,2.405554,0.368975
4,116.014,141.781,110.655,0.01284,0.00011,0.00655,0.00908,0.01966,0.06425,0.584,...,0.10470,0.01767,19.649,1,0.417356,0.823484,-3.747787,0.234513,2.332180,0.410335


In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 195 entries, 0 to 194
Data columns (total 23 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   MDVP:Fo(Hz)       195 non-null    float64
 1   MDVP:Fhi(Hz)      195 non-null    float64
 2   MDVP:Flo(Hz)      195 non-null    float64
 3   MDVP:Jitter(%)    195 non-null    float64
 4   MDVP:Jitter(Abs)  195 non-null    float64
 5   MDVP:RAP          195 non-null    float64
 6   MDVP:PPQ          195 non-null    float64
 7   Jitter:DDP        195 non-null    float64
 8   MDVP:Shimmer      195 non-null    float64
 9   MDVP:Shimmer(dB)  195 non-null    float64
 10  Shimmer:APQ3      195 non-null    float64
 11  Shimmer:APQ5      195 non-null    float64
 12  MDVP:APQ          195 non-null    float64
 13  Shimmer:DDA       195 non-null    float64
 14  NHR               195 non-null    float64
 15  HNR               195 non-null    float64
 16  status            195 non-null    int64  
 1

In [16]:
df.drop('name', axis = 1, inplace = True)

In [17]:
x = df.drop('status', axis = 1)
y = df['status']

In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

In [19]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [21]:
model = RandomForestClassifier(random_state=42, class_weight = 'balanced_subsample')
model.fit(X_train, y_train)
pred = model.predict(X_test)

In [22]:
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.89      0.80      0.84        10
           1       0.93      0.97      0.95        29

    accuracy                           0.92        39
   macro avg       0.91      0.88      0.90        39
weighted avg       0.92      0.92      0.92        39



In [31]:
X_train_tensor = torch.tensor(X_train, dtype = torch.float32)
X_test_tensor = torch.tensor(X_test, dtype = torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype = torch.long)
y_test_tensor = torch.tensor(y_test.values, dtype = torch.long)

In [32]:
# 모델 정의
class parkinsons_clf(nn.Module):
    def __init__(self, _dim):
        super(parkinsons_clf, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(_dim, _dim*3),    
            nn.ReLU(),                  
            nn.Linear(_dim*3, _dim*2),
            nn.ReLU(),                 
            nn.Linear(_dim*2, 4)
        )
    def forward(self, x):
        return self.model(x)

In [33]:
parkinsons_model = parkinsons_clf(X_train_tensor.shape[1])

In [34]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(parkinsons_model.parameters(), lr = 0.01)

In [35]:
for epoch in range(150):
    pred = parkinsons_model(X_train_tensor)
    loss = criterion(pred, y_train_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch+1) % 30 == 0:
        print(f"Epoch {epoch+1}, Loss: {round(loss.item(), 6)}")

Epoch 30, Loss: 0.113455
Epoch 60, Loss: 0.013761
Epoch 90, Loss: 0.002607
Epoch 120, Loss: 0.000864
Epoch 150, Loss: 0.000465


In [36]:
parkinsons_model.eval()
with torch.no_grad():
    pred = parkinsons_model(X_test_tensor)
    _, pred_idx = torch.max(pred, 1)
    acc = accuracy_score(y_test_tensor, pred_idx)
    print(f"Test Accuracy : {round(acc, 4)}")
    print(classification_report(y_test_tensor, pred_idx))

Test Accuracy : 0.9231
              precision    recall  f1-score   support

           0       0.82      0.90      0.86        10
           1       0.96      0.93      0.95        29

    accuracy                           0.92        39
   macro avg       0.89      0.92      0.90        39
weighted avg       0.93      0.92      0.92        39

